# BERTopic clustering with BGE-M3, UMAP, and HDBSCAN

This notebook mirrors the aggregation pipeline pattern used in the other notebooks, but uses `BAAI/bge-m3` embeddings inside a BERTopic workflow. UMAP reduces the embedding space, HDBSCAN discovers dense event/topic clusters, and `CountVectorizer` uses a trilingual stopword list to suppress boilerplate vocabulary while preserving meaningful Sinhala, Tamil, and English terms.

In [1]:
# Install dependencies when running in a fresh notebook environment.
# Restart the kernel if any package is newly installed.
!pip install -q bertopic sentence-transformers umap-learn hdbscan grapheme
!pip install -U "scipy>=1.15.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 57.7 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires scipy<1.17,>=1.8, but you have scipy 1.18.0 which is incompatible.


In [2]:
from pathlib import Path
import os
import re
import unicodedata

import numpy as np
import pandas as pd
import grapheme

SEED = 42
np.random.seed(SEED)

NOTEBOOK_DIR = Path.cwd()

# Dataset location
DATA_DIR = Path("/kaggle/input/datasets/uom230425m/aggregation-pipeline-data/data")

# Writable location for results
RESULTS_ROOT = Path("/kaggle/working/results")
RESULTS_DIR = RESULTS_ROOT / "bge_m3_bertopic"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data dir: {DATA_DIR.resolve()}")
print(f"Results dir: {RESULTS_DIR.resolve()}")

Data dir: /kaggle/input/datasets/uom230425m/aggregation-pipeline-data/data
Results dir: /kaggle/working/results/bge_m3_bertopic


In [3]:
# Load annotator files
df_a = pd.read_csv(DATA_DIR / "a.csv")
df_d = pd.read_csv(DATA_DIR / "d.csv")
df_p = pd.read_csv(DATA_DIR / "p.csv")

for frame in (df_a, df_d, df_p):
    frame.drop(columns=["bias_label"], inplace=True, errors="ignore")

print(df_a.shape, df_d.shape, df_p.shape)

(750, 6) (750, 8) (800, 6)


In [4]:
# Concatenate dataframes and drop duplicate articles by article_id
df = pd.concat([df_a, df_d, df_p], ignore_index=True, sort=False)
df = df.drop_duplicates(subset="article_id", keep="first").reset_index(drop=True)
df.drop(columns=["flags", "Unnamed: 8"], inplace=True, errors="ignore")

print("df shape:", df.shape)
df.head()

df shape: (2000, 6)


,article_id,publisher,url,published_at,title,body_text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...


In [5]:
# Keep rows with the fields required for clustering
required_columns = ["article_id", "publisher", "url", "published_at", "title", "body_text"]
df = df.dropna(subset=required_columns).copy()

print("df shape after required-field dropna:", df.shape)
df.info()

df shape after required-field dropna: (1999, 6)
<class 'pandas.core.frame.DataFrame'>
Index: 1999 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   article_id    1999 non-null   object
 1   publisher     1999 non-null   object
 2   url           1999 non-null   object
 3   published_at  1999 non-null   object
 4   title         1999 non-null   object
 5   body_text     1999 non-null   object
dtypes: object(6)
memory usage: 109.3+ KB


In [6]:
# Unicode normalization (NFC - canonical decomposition + composition)
for column in ["title", "body_text"]:
    df[column] = df[column].astype(str).map(lambda value: unicodedata.normalize("NFC", value))

df.head()

,article_id,publisher,url,published_at,title,body_text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...


In [7]:
# Remove title duplication from the beginning of the body text
def remove_title_from_body(row):
    body = row["body_text"].strip()
    title = row["title"].strip()
    if body.startswith(title):
        body = body[len(title):].lstrip("\n").lstrip()
    return body


df["text"] = df.apply(remove_title_from_body, axis=1)
df[["title", "text"]].head()

,title,text
0,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් හාල්මැස්සො නොව තෝරු මෝරු අහුවෙන කාලය බවමහ...
1,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',"මෙහි කිසිවක් අඩංගු නැත.Play video, ""දිරිය මිනි..."
2,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,සිගිරිය පිදුරංගල මාර්ගයේදී ඊයේ සවස වන අලියෙකු ...
3,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් දෙ...
4,ස්ථාන දෙකකදී ඝාතන දෙකක්,කුලී නිවසක පදිංචිව සිටි පුද්ගලයෙකුව ඊයේ (18) ර...


In [28]:
print(df["text"][10])

2025 රාත්‍රී කාලයේ ගල්නෑව පොලිස් වසමේ බුල්නෑව ප්‍රදේශයේ නිවසක් රුපියල් පන්කෝටි හැත්තැදෙලක්ෂ හැටඅටදහාක් 268 000 වටිනා ප්‍රාඩෝ වර්ගයේ රථයක් භාණ්ඩ මුදල් ජංගම දුරකථන කොල්ලකෑමක් සම්බන්ධයෙන් ගල්නෑව පොලිස් ස්ථානයට පැමිණිල්ලක් විමර්ශන ආරම්භ තිබුණි 2025 ගල්නෑව පොලිස් ස්ථානයේ නිලධාරීන් විසින් ඉහත අපරාධයට සම්බන්ධ සැකකරුවන් හයදෙනෙකු කුරුණෑගල මාතවතගම විල්ගමුව ප්‍රදේශවලදී හෙරොයින් ග්‍රෑම් මිලිග්‍රෑම් 300 අයිස් මත්ද්‍රව්‍ය ග්‍රෑම් මිලිග්‍රෑම් 250 අපරාධයට යොදාගත් අත්වැසුම් අපරාධයට යතුරුපැදියක් රථයක් රථයක් සමඟ අත්අඩංගුවට සැකකරුවන් අවුරුදු වයස්වල පසුවන කුරුණෑගල නුගගොල්ල මාවතගම මැල්සිරිපුර මහනුවර ප්‍රදේශවල පදිංචිකරුවන් සැකකරුවන් විසින් කොල්ලකන ප්‍රාඩෝ වර්ගයේ රථය මීගලෑව ප්‍රදේශයේ තිබියදී සොයාගෙන තවද කොල්ලකන මුදලින් රුපියල් පනස්නමලක්ෂ තිස්නමදාහක 939 000 මුදලක් කොල්ලකන මුදල් වලින් උපයාගන්නා දේපළ රැසක් විමර්ශන නිලධාරීන් භාරයට සැකකරුවන් 2025 කැකිරාව මහේස්ත්‍රාත් අධිකරණයට ඉදිරිපත් වැඩිදුර විමර්ශන සඳහා 2025 දක්වා රැඳවුම් නියෝග ලබාගෙන ගල්නෑව පොලිසිය වැඩිදුර විමර්ශන කරනු ලබයි


In [8]:
import unicodedata

def clean_text(text):
    # Replace punctuation with spaces
    text = "".join(
        " " if unicodedata.category(char).startswith("P") else char
        for char in text
    )

    # Keep words with more than 2 graphemes
    return " ".join(
        word for word in text.split()
        if len(list(grapheme.graphemes(word))) > 2
    )

df["text"] = df["text"].apply(clean_text)

In [25]:
print(df["text"][10])

2025 රාත්‍රී කාලයේ ගල්නෑව පොලිස් වසමේ බුල්නෑව ප්‍රදේශයේ නිවසක් රුපියල් පන්කෝටි හැත්තැදෙලක්ෂ හැටඅටදහාක් 268 000 වටිනා ප්‍රාඩෝ වර්ගයේ රථයක් භාණ්ඩ මුදල් ජංගම දුරකථන කොල්ලකෑමක් සම්බන්ධයෙන් ගල්නෑව පොලිස් ස්ථානයට පැමිණිල්ලක් විමර්ශන ආරම්භ තිබුණි 2025 ගල්නෑව පොලිස් ස්ථානයේ නිලධාරීන් විසින් ඉහත අපරාධයට සම්බන්ධ සැකකරුවන් හයදෙනෙකු කුරුණෑගල මාතවතගම විල්ගමුව ප්‍රදේශවලදී හෙරොයින් ග්‍රෑම් මිලිග්‍රෑම් 300 අයිස් මත්ද්‍රව්‍ය ග්‍රෑම් මිලිග්‍රෑම් 250 අපරාධයට යොදාගත් අත්වැසුම් අපරාධයට යතුරුපැදියක් රථයක් රථයක් සමඟ අත්අඩංගුවට සැකකරුවන් අවුරුදු වයස්වල පසුවන කුරුණෑගල නුගගොල්ල මාවතගම මැල්සිරිපුර මහනුවර ප්‍රදේශවල පදිංචිකරුවන් සැකකරුවන් විසින් කොල්ලකන ප්‍රාඩෝ වර්ගයේ රථය මීගලෑව ප්‍රදේශයේ තිබියදී සොයාගෙන තවද කොල්ලකන මුදලින් රුපියල් පනස්නමලක්ෂ තිස්නමදාහක 939 000 මුදලක් කොල්ලකන මුදල් වලින් උපයාගන්නා දේපළ රැසක් විමර්ශන නිලධාරීන් භාරයට සැකකරුවන් 2025 කැකිරාව මහේස්ත්‍රාත් අධිකරණයට ඉදිරිපත් වැඩිදුර විමර්ශන සඳහා 2025 දක්වා රැඳවුම් නියෝග ලබාගෙන ගල්නෑව පොලිසිය වැඩිදුර විමර්ශන කරනු ලබයි


In [26]:
# Whitespace normalization
df["text"] = df["text"].map(lambda value: re.sub(r"\n{2,}", "\n", value))
df["text"] = df["text"].map(lambda value: re.sub(r"[ \t]+", " ", value))
df["text"] = df["text"].str.strip()

df = df[df["text"].str.len() > 0].reset_index(drop=True)
print("df shape after text cleaning:", df.shape)
df.head()
print(df["text"][10])

df shape after text cleaning: (1999, 10)
2025 රාත්‍රී කාලයේ ගල්නෑව පොලිස් වසමේ බුල්නෑව ප්‍රදේශයේ නිවසක් රුපියල් පන්කෝටි හැත්තැදෙලක්ෂ හැටඅටදහාක් 268 000 වටිනා ප්‍රාඩෝ වර්ගයේ රථයක් භාණ්ඩ මුදල් ජංගම දුරකථන කොල්ලකෑමක් සම්බන්ධයෙන් ගල්නෑව පොලිස් ස්ථානයට පැමිණිල්ලක් විමර්ශන ආරම්භ තිබුණි 2025 ගල්නෑව පොලිස් ස්ථානයේ නිලධාරීන් විසින් ඉහත අපරාධයට සම්බන්ධ සැකකරුවන් හයදෙනෙකු කුරුණෑගල මාතවතගම විල්ගමුව ප්‍රදේශවලදී හෙරොයින් ග්‍රෑම් මිලිග්‍රෑම් 300 අයිස් මත්ද්‍රව්‍ය ග්‍රෑම් මිලිග්‍රෑම් 250 අපරාධයට යොදාගත් අත්වැසුම් අපරාධයට යතුරුපැදියක් රථයක් රථයක් සමඟ අත්අඩංගුවට සැකකරුවන් අවුරුදු වයස්වල පසුවන කුරුණෑගල නුගගොල්ල මාවතගම මැල්සිරිපුර මහනුවර ප්‍රදේශවල පදිංචිකරුවන් සැකකරුවන් විසින් කොල්ලකන ප්‍රාඩෝ වර්ගයේ රථය මීගලෑව ප්‍රදේශයේ තිබියදී සොයාගෙන තවද කොල්ලකන මුදලින් රුපියල් පනස්නමලක්ෂ තිස්නමදාහක 939 000 මුදලක් කොල්ලකන මුදල් වලින් උපයාගන්නා දේපළ රැසක් විමර්ශන නිලධාරීන් භාරයට සැකකරුවන් 2025 කැකිරාව මහේස්ත්‍රාත් අධිකරණයට ඉදිරිපත් වැඩිදුර විමර්ශන සඳහා 2025 දක්වා රැඳවුම් නියෝග ලබාගෙන ගල්නෑව පොලිසිය වැඩිදුර විමර්ශන කරනු ල

In [27]:
# Build documents for semantic embedding and topic representation
def build_document(title, body):
    title = str(title).strip()
    body = str(body).strip()
    return f"{title}. {body}" if title else body


df["document"] = df.apply(lambda row: build_document(row["title"], row["text"]), axis=1)
documents = df["document"].tolist()

print(f"Documents: {len(documents)}")
documents[0][:500]
print(df["text"][10])

Documents: 1999
2025 රාත්‍රී කාලයේ ගල්නෑව පොලිස් වසමේ බුල්නෑව ප්‍රදේශයේ නිවසක් රුපියල් පන්කෝටි හැත්තැදෙලක්ෂ හැටඅටදහාක් 268 000 වටිනා ප්‍රාඩෝ වර්ගයේ රථයක් භාණ්ඩ මුදල් ජංගම දුරකථන කොල්ලකෑමක් සම්බන්ධයෙන් ගල්නෑව පොලිස් ස්ථානයට පැමිණිල්ලක් විමර්ශන ආරම්භ තිබුණි 2025 ගල්නෑව පොලිස් ස්ථානයේ නිලධාරීන් විසින් ඉහත අපරාධයට සම්බන්ධ සැකකරුවන් හයදෙනෙකු කුරුණෑගල මාතවතගම විල්ගමුව ප්‍රදේශවලදී හෙරොයින් ග්‍රෑම් මිලිග්‍රෑම් 300 අයිස් මත්ද්‍රව්‍ය ග්‍රෑම් මිලිග්‍රෑම් 250 අපරාධයට යොදාගත් අත්වැසුම් අපරාධයට යතුරුපැදියක් රථයක් රථයක් සමඟ අත්අඩංගුවට සැකකරුවන් අවුරුදු වයස්වල පසුවන කුරුණෑගල නුගගොල්ල මාවතගම මැල්සිරිපුර මහනුවර ප්‍රදේශවල පදිංචිකරුවන් සැකකරුවන් විසින් කොල්ලකන ප්‍රාඩෝ වර්ගයේ රථය මීගලෑව ප්‍රදේශයේ තිබියදී සොයාගෙන තවද කොල්ලකන මුදලින් රුපියල් පනස්නමලක්ෂ තිස්නමදාහක 939 000 මුදලක් කොල්ලකන මුදල් වලින් උපයාගන්නා දේපළ රැසක් විමර්ශන නිලධාරීන් භාරයට සැකකරුවන් 2025 කැකිරාව මහේස්ත්‍රාත් අධිකරණයට ඉදිරිපත් වැඩිදුර විමර්ශන සඳහා 2025 දක්වා රැඳවුම් නියෝග ලබාගෙන ගල්නෑව පොලිසිය වැඩිදුර විමර්ශන කරනු ලබයි


In [24]:
print("Sample document:", documents[10:50])

Sample document: ['බුල්නෑව නිවසක් බිඳ ජීප් රථයක්, රන් භාණ්ඩ හා මුදල් කෝල්ල කෑ සැකකරුවන් අල්ලයි. 2025 රාත්\u200dරී කාලයේ ගල්නෑව පොලිස් වසමේ බුල්නෑව ප්\u200dරදේශයේ නිවසක් රුපියල් පන්කෝටි හැත්තැදෙලක්ෂ හැටඅටදහාක් 268 000 වටිනා ප්\u200dරාඩෝ වර්ගයේ රථයක් භාණ්ඩ මුදල් ජංගම දුරකථන කොල්ලකෑමක් සම්බන්ධයෙන් ගල්නෑව පොලිස් ස්ථානයට පැමිණිල්ලක් විමර්ශන ආරම්භ තිබුණි 2025 ගල්නෑව පොලිස් ස්ථානයේ නිලධාරීන් විසින් ඉහත අපරාධයට සම්බන්ධ සැකකරුවන් හයදෙනෙකු කුරුණෑගල මාතවතගම විල්ගමුව ප්\u200dරදේශවලදී හෙරොයින් ග්\u200dරෑම් මිලිග්\u200dරෑම් 300 අයිස් මත්ද්\u200dරව්\u200dය ග්\u200dරෑම් මිලිග්\u200dරෑම් 250 අපරාධයට යොදාගත් අත්වැසුම් අපරාධයට යතුරුපැදියක් රථයක් රථයක් සමඟ අත්අඩංගුවට සැකකරුවන් අවුරුදු වයස්වල පසුවන කුරුණෑගල නුගගොල්ල මාවතගම මැල්සිරිපුර මහනුවර ප්\u200dරදේශවල පදිංචිකරුවන් සැකකරුවන් විසින් කොල්ලකන ප්\u200dරාඩෝ වර්ගයේ රථය මීගලෑව ප්\u200dරදේශයේ තිබියදී සොයාගෙන තවද කොල්ලකන මුදලින් රුපියල් පනස්නමලක්ෂ තිස්නමදාහක 939 000 මුදලක් කොල්ලකන මුදල් වලින් උපයාගන්නා දේපළ රැසක් විමර්ශන නිලධාරීන් භාරයට සැකකරුවන් 2025 කැකිරාව ම

In [ ]:


stopwords = {
    "අද", "අප", "අපි", "අපේ", "අතර", "අනුව", "අය", "අයගේ", "ඇත", "ඇති", "ඇතුළු",
    "ඒ", "එම", "එය", "එහි", "එක්", "එක", "එකක්", "කර", "කරන", "කරයි", "කරමින්",
    "කළ", "කිරීම", "කියන", "කියා", "ගැන", "ගැනීමට", "ගැනීම", "තම", "තුළ", "ද", "දී",
    "දක්වා", "දැයි", "නම්", "නමුත්", "නිසා", "බව", "බවට", "බවයි", "බවද", "මත", "මෙන්",
    "මෙම", "මේ", "ය", "යන", "ලෙස", "වූ", "වෙත", "වෙනුවෙන්", "සහ", "සිට", "සඳහා",
    "හා", "හෝ", "මගින්", "බවත්"
}



stopwords = sorted(stopwords)

print(f"Stopwords: {len(stopwords)}")

Stopwords: 57


In [15]:
# Embed documents with BGE-M3
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model_name = "BAAI/bge-m3"
embedding_model = SentenceTransformer(embedding_model_name, device=device)

embeddings = embedding_model.encode(
    documents,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print(embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

(1999, 1024)


In [ ]:
# Configure BERTopic with UMAP, HDBSCAN, and CountVectorizer
from bertopic import BERTopic
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

umap_model = UMAP(
    n_neighbors=3,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=SEED,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=2,
    min_samples=1,
    metric="euclidean",
    cluster_selection_method="leaf",
    prediction_data=True,
)

vectorizer_model = CountVectorizer(
    stop_words=list(stopwords),
    token_pattern=r"(?u)\b\w\w+\b",
    ngram_range=(1, 2),
    min_df=1,
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language="multilingual",
    calculate_probabilities=True,
    verbose=True,
)

In [56]:
# Fit BERTopic using the precomputed BGE-M3 embeddings
topics, probabilities = topic_model.fit_transform(documents, embeddings)

df["topic_id"] = topics
df["topic_probability"] = [float(np.max(row)) if row is not None and len(row) else np.nan for row in probabilities]

topic_info = topic_model.get_topic_info()
topic_info.head(20)

2026-08-10 15:49:03,361 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


2026-08-10 15:49:11,729 - BERTopic - Dimensionality - Completed ✓
2026-08-10 15:49:11,730 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-10 15:49:11,843 - BERTopic - Cluster - Completed ✓
2026-08-10 15:49:11,848 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-10 15:49:13,383 - BERTopic - Representation - Completed ✓


TypeError: object of type 'numpy.float64' has no len()

In [51]:
# Cluster/topic summary
print(df["topic_id"].value_counts().head(20))
print("Noise articles:", int((df["topic_id"] == -1).sum()))
print("Number of topics found:", df["topic_id"].nunique() - (1 if -1 in df["topic_id"].values else 0))

topic_info.head(30)

topic_id
-1     343
 0      17
 1      13
 2      13
 3      12
 4      12
 5      12
 6      12
 7      12
 8      12
 9      11
 10     11
 11     11
 12     11
 13     11
 14     11
 15     11
 16     11
 17     10
 18     10
Name: count, dtype: int64
Noise articles: 343
Number of topics found: 276


,Topic,Count,Name,Representation,Representative_Docs
0,-1,343,-1_වන_සම_නව_යල,"[වන, සම, නව, යල, ලක, මට, පත, අන, හන, වත]",[ඉදිරි ජනාධිපතිවරණයේදී ඡන්දදායකයින් ලක්ෂ 7කට ඡ...
1,0,17,0_මන වර_මන_වරප_වරප රස,"[මන වර, මන, වරප, වරප රස, මන වරය, පක කත, පනත, අ...",[පක්ෂ විනය කඩ කිරීම නිසා අධිකරණ නියෝග වලින් මන...
2,1,13,1_පනත_පනත පත_බල පනත_අඟ,"[පනත, පනත පත, බල පනත, අඟ, බල, twitter, බලශක, අ...",[ශ්‍රී ලංකාව ඉන්දියාවේ කොටසක් ද? :'හරින් ඇමතික...
3,2,13,2_ඉර_ඉර නය_ජනපදය_සත ජනපදය,"[ඉර, ඉර නය, ජනපදය, සත ජනපදය, සම සන, ඇම, කච, නල...",[ඇතැම් නැව් ආරක්ෂිතව ගමන් කරවීමට චීනය හා ඉරානය...
4,3,12,3_ඩකය_කට_තරග_කට ආයතනය,"[ඩකය, කට, තරග, කට ආයතනය, තරග වල, කණ, ටර වන, කණ...",[Fox Hill අනතුර: සාමාන්‍ය මාර්ග නීති මෝටර් රථ ...
5,4,12,4_ජනත රම_ආර ගම_ඊශ යල_ඇම,"[ජනත රම, ආර ගම, ඊශ යල, ඇම, යල කය, ඊශ, භය ජපක, ...",[ආරුගම්බේ ආරක්ෂක තර්ජනය ගැන ඇමති හෙළි කළ කරුණු...
6,5,12,5_නවකත_වන දය_චර_දය,"[නවකත, වන දය, චර, දය, නවකත වක, යනව, පටන, ශල, ත...",[මම ලියන හැටි' : රූපා ශ්‍රියානි ඒකනායක. ශ්‍රිය...
7,6,12,6_තනය_ආය ධයක_ගලය_බළ,"[තනය, ආය ධයක, ගලය, බළ, වසක, පහරද තනය, කකර, පහර...",[තියුණු ආයුධයකින් පහරදී පුද්ගලයෙකු ඝාතනය කරයි....
8,7,12,7_වන හම_කසන_කන කසන_කට යක,"[වන හම, කසන, කන කසන, කට යක, කසන රය, යක වන, ඉන,...",[ආධාර රැගත් තවත් යානයක් කටුනායකට. ලාංකික ජනතාව...
9,8,12,8_නද_cctv_නද උත_ගලය,"[නද, cctv, නද උත, ගලය, පහරද, වන සර, හලට, හලට ළ...",[වනාතේ බිම්සරට එල්ල වූ වෙඩිල්ල. බොරැල්ලේ පිහිට...


In [52]:
# Inspect sample titles per topic
for topic_id, group in df[df["topic_id"] != -1].groupby("topic_id"):
    print(f"=== Topic {topic_id} ({len(group)} articles) ===")
    for title in group["title"].head(10):
        print(f"- {title}")
    print()

    if topic_id >= 10:
        break

=== Topic 0 (17 articles) ===
- මහනුවර නගර සභා මන්ත්‍රීන්ගේ දුරකතන දීමනාව අවලංගුයි
- තවත් කලු සුද්දෝ පාර්ලිමේන්තුවේ ඉන්නවා
- මහ බැංකුවේ අත්තනෝමතික වැටුප් වැඩි කර ගැනීම අවලංගු කිරීමේ පනත පාර්ලිමේන්තුවට
- ධීවර විශ්‍රාම වැටුප යථාර්ථයක් වෙන හැඩ
- පාර්ලිමේන්තු විශ්‍රාම වැටුප් පනත ඉවත් කිරීමට කැබිනට් අනුමැතිය
- මන්ත්‍රී විශ්‍රාම වැටුප ඉවත් කරන පනත් කෙටුම්පත පාර්ලිමේන්තුවට: 400කට වැඩි ප්‍රතිලාභීන්ට සිදුවන්නේ කුමක් ද?
- අමාත්‍යවරයෙකුගේ මන්ත්‍රී ධුරය බල රහිත කරන ලෙස ඉල්ලා අභියාචනාධිකරණයට පෙත්සමක්
- පැන්ෂන් කැපුවොත් නඩු දානවා
- මන්ත්‍රීවරුන්ට ගිනිඅවි දීම: ආණ්ඩුව සැරසෙන්නේ 'මොකක් හරි සෙල්ලමකට' ද?
- මන්ත්‍රී පුටුවක් හිස්: මහ ලේකම් මැකෝට  දන්වයි

=== Topic 1 (13 articles) ===
- මතභේදයට තුඩු දී ඇති විදුලිබල පනත් කෙටුම්පත ආණ්ඩුක්‍රම ව්‍යවස්ථාවට පටහැනි වන්නේ කෙසේද ?
- පාර්ලිමේන්තු මන්ත්‍රීවරුන්ට පෙර වාහන පර්මිට් හිමිවූ රාජ්‍ය නිලධාරින් කවුද ?
- රියදුරු බලපත්‍ර එක් දින සේවාව පිටපළාත්වලටත්
- විදුලිබල පනත් කෙටුම්පත සම්මතයි: දැන් සිදුවන්නේ කුමක් ද?
- ගල් අඟුරු ටෙන්ඩරයෙන්  දූෂණයකට  පාර කැපිලා
- විදුලියේ වර

In [20]:
# Save article-level assignments and topic summaries
assignments_path = RESULTS_DIR / "bge_m3_bertopic_assignments.csv"
topics_path = RESULTS_DIR / "bge_m3_bertopic_topics.csv"

df.drop(columns=["document"], errors="ignore").to_csv(assignments_path, index=False, encoding="utf-8-sig")
topic_info.to_csv(topics_path, index=False, encoding="utf-8-sig")

print(f"Saved assignments: {assignments_path.resolve()}")
print(f"Saved topics: {topics_path.resolve()}")

Saved assignments: /kaggle/working/results/bge_m3_bertopic/bge_m3_bertopic_assignments.csv
Saved topics: /kaggle/working/results/bge_m3_bertopic/bge_m3_bertopic_topics.csv


In [21]:
# Optional BERTopic visualizations
# These are useful in Jupyter/Colab but can be slow for large corpora.

# topic_model.visualize_topics()
# topic_model.visualize_barchart(top_n_topics=20)
# topic_model.visualize_hierarchy()